## **CONCEPTOS**

### **Escalabilidad**

- **Definición:** Se refiere a la capacidad del sistema para manejar un incremento en la carga de trabajo o en el volumen de datos al añadir recursos adicionales al sistema existente, como nodos de procesamiento o almacenamiento.

- **Ejemplo:** Si un cluster de Big Data es escalable, significa que puede crecer de manera eficiente agregando más nodos de procesamiento o almacenamiento según sea necesario para manejar mayores volúmenes de datos o demandas de computación.

En resumen, la escalabilidad se enfoca en la capacidad de crecer al agregar recursos de manera manual o planificada.

### **Elasticidad**

- **Definición:** Es la capacidad de un sistema para adaptarse dinámicamente a las demandas cambiante de la carga de trabajo. Esto implica la capacidad de agregar o remover recursos automáticamente en respuesta a cambios en la carga, de manera que se optimice el rendimiento y se minimicen los costos.

 - **Ejemplo:** Un cluster de Big Data que es elástico ajusta automáticamente la cantidad de nodos de procesamiento o almacenamiento activos según las necesidades actuales de procesamiento o almacenamiento. Por ejemplo, puede aumentar el número de nodos durante las horas pico y reducirlos durante períodos de menor actividad, todo de forma automática.

En resumen, la elasticidad se refiere a la capacidad de ajustarse automáticamente a las fluctuaciones de la carga de trabajo sin intervención manual, optimizando así el rendimiento y los costos operativos.

### **Herramientas Cloud equivalentes**

<center><img src="https://i.postimg.cc/fLnLVsFQ/p74.png"></center>

### **Formato de archivos**

#### **`¿Que tipo de formato BINARIO escojo?`**

En HIVE existen dos tipos de formatos validos:

- **FORMATOS ESTRUCTURADOS**     : Aquellos archivos en los cuales existe una variabilidad baja en el numero de sus 
                                campos.

- **FORMATOS SEMI-ESTRUCTURADOS** : Aquellos archivos en los cuales existe mucha variabilidad en el numero de sus campos.
                                Por ejemplo, periodicamente podriamos tener la llegada de archivos, para los cuales 
                                varia el numero de sus campos, es decir, el dia 1 nos llega el archivo con los campos
                                'nombre','apellido','edad','ciudad', sin embargo, para el dia 2 nos llega el archivo
                                solo con los campos 'nombre' y 'edad', y asi sucesivamente, donde para los archivos 
                                su numero de campos VARIA DE FORMA RECURRENTE.

&rarr; Al trabajar con archivos **SEMI-ESTRUCTURADOS** se recomienda utilizar el formato binario `AVRO`. Si la variabilidad de
campos es ALTA debemos formatear nuestros archivos al formato AVRO.                                

&rarr; Al trabajar con archivos **ESTRUCTURADOS** se recomienda utilizar los formatos binarios `PARQUET` y `ORC`. Sin embargo PARQUET no es compatible con el tipo de dato DATE y el formato ORC si lo es. Por otro lado, PARQUET puede llegar a ser 10 veces mas rapido que ORC.

#### **`Existen 4 formatos de archivo:`**

&rarr; **TEXTFILE**: Formato de texto plano. El formato TEXTFILE es el peor formato que existe para procesamiento, es muy lento. Es bueno porque nos permite colocar con un "put" la data sobre HDFS, malo porque es muy lento al procesar (por ejemplo, para operaciones JOIN y GROUP BY). Generalmente el formato TEXTFILE se utiliza SOLO PARA REALIZAR LA INGESTA DE DATOS, pero esta PROHIBIDO lanzar cualquier proceso sobre estas tablas. Tan simple como un COUNT o tan complejo como una Red Neuronal. 

&rarr; **PARQUET**: Formato binario, bueno porque es el formato más rápido para procesamiento, malo porque en algunas versiones no se soporta el tipo "DATE".

&rarr; **ORC**: Formato binario, bueno porque es el segundo formato más rápido para procesamiento y soportar tipos de datos DATE, malo porque no es compatible con otras herramientas como IMPALA.

&rarr; **AVRO**: Formato binario, bueno porque permite tener un esquema dinámico que varia en el tiempo, malo porque tiene un procesamiento lento.

#### **`Los archivos de texto plano no guardan la metadata y los archivos binarios si lo hacen`**

Si guardamos la información (en un dataframe) de un archivo de texto plano y luego la recuperamos y consultamos el esquema de metadatos, nos devolverá todo como tipo de datos **STRING**. `Esto no sucede con los archivos BINARIOS`. Los archivos binarios guardan la metadata.

### **Compresión de archivos**

#### **SNAPPY**

**¿En cuánto puede llegar a comprimir SNAPPY?** en el mejor de los casos puede comprimir los datos hasta en un 80% y recuerda que estamos en un entorno de Big data, si estamos en gestando una tabla de datos de 1 TB, al comprimir con SNAPPY, ese terabyte que son 1.000 GB de datos, podemos reducirlo a 200 GB de datos y nos estamos ahorrando 800 GB, eso es bastante información. Por supuesto que la compresión tiene una penalización, las consultas van a correr un poquito más lentas de que si solo estuvieran, por ejemplo, en AVRO, pero miren la ganancia en tiempo y ahorro de disco duro. El estándar en el caso de procesos batch, la data que sea batchera se debe comprimir, pero la data que sea en tiempo real, esa sí no la vamos a comprimir, porque, el proceso se va a hacer más lento si lo comprimimos, entonces, ahí no se comprime.

#### **Compresión en un archivo de formato AVRO v/s un archivo formato PARQUET**

Los archivos de formato AVRO al guardalos con compresión, por ejemplo, 'snappy', la partición no lo va a indicar, no asi, 
como una particion de formato PARQUET:
```scala
dfResultado.write.format("avro"). \
            mode("overwrite"). \
            option("compression", "snappy"). \
            save("dbfs:///FileStore/_spark/output/dfResultadoAvro")

Resultado: part-00000-tid-953579043164171325-b11d8b33-b690-4093-a17f-04a8d66b19e5-40-1-c000.avro

dfResultado.write.format("parquet"). \
            mode("overwrite"). \
            option("compression", "snappy"). \
            save("dbfs:///FileStore/_spark/output/dfResultadoParquet")

Resultado: part-00000-tid-4278820346301987657-30a2a881-3b6e-40f4-8dc6-48bea272446e-36-1-c000.snappy.parquet
```

**¿Por qué sucede esto?**

Esto se debe a que el formato binario AVRO por defecto comprime los datos, entonces, como por defecto comprime los datos, no tiene sentido que agregue la extensión '.snappy', dado que cualquier cosa en AVRO, YA ESTÁ COMPRIMIDO.

### **Gobierno de datos**

#### **`DATA LAKE`**

##### **LANDING_TMP**

Es una capa de aterrizaje, **si son archivos estructurados van a vivir en tablas**, **si son archivos semiestructurados los colocaremos en un directorio**, **si son archivos no estructurados como imágenes también irán en un directorio**. Así es como la estrategia de captura de datos ingesta los datos en una capa llamada LANDING TEMPORAL. Ahora, el objetivo de la capa de LANDING TEMPORAL es capturar la data del día de hoy.

##### **LANDING**

Hay una capa llamada LANDING que tiene el objetivo de ir acumulando los datos, por ejemplo, en esta capa **para los archivos estructurados tenemos una tabla que va a estar particionada por la fecha en la que estamos ingestando los datos** y aquí tendremos el archivo de anteayer, está el archivo de ayer y el archivo de hoy. `Luego, al día siguiente se borrará el archivo que se encuentra en la capa LANDING_TMP y se ingresará este nuevo archivo a esta capa y lo moveremos a la capa LANDING a la tabla en la partición correspondiente y de esa manera vamos a ir acumulando los datos`. Eso se hace en la zona de LANDING. **Para los archivos semi-estructurados y no estructurados** vamos a tener un directorio en la capa LANDING, digamos que estamos capturando un archivo JSON, dentro de este directorio habrá un subdirectorio que dirá: " … esta es la fecha del día de anteayer, luego esta es el archivo de la fecha del día de ayer y el día de hoy creamos un nuevo directorio y guardamos el archivo así y de esa manera vamos capturando los datos … ". `Es decir, se crearon dentro del directorio, distintos subdirectorios para cada fecha`. Lo mismo con los datos no estructurados, un directorio con subdirectorios y cada subdirectorio va a tener las imágenes, vídeos o lo que se esté capturando ese día. Por eso esta capa se conoce como LANDING TEMPORAL es solamente para escribir los datos desde las fuentes de datos, eso que pusimos en la estrategia de ingesta de los datos, eso escribe aquí, pero luego tenemos que moverlo a las particiones o a los subdirectorios de manera **MANUAL**. Adicionalmente a eso, **HAY QUE BINARIZAR LOS DATOS**. 

##### **UNIVERSAL**

Una vez que esté en la zona de LANDING hay que limpiar esos datos, porque, pueden venir con errores. Eso se hace en una capa conocida como **UNIVERSAL**. Implementaremos aquí un proceso que se conecte a la data que se está ingestando el día de hoy y le aplique las reglas de calidad y eso como resultante nos entregará una tabla limpia, si estamos ingresando la tabla 'Persona' en UNIVERSAL tendremos también la tabla 'Persona' pero con registros limpios. Lo mismo va a pasar con los datos de semi estructurados y no estructurados, pero, acá hay un nivel de complejidad que vamos a explicar en unos momentos luego de entender un poquito mejor esto. **Lo importante es que en UNIVERSAL ya vive la data de manera limpia y lista para ser consultada por las soluciones**, eso se hace en la zona de SMART.

##### **DATA_MESH**

Ahora ¿qué es lo que pasa? antiguamente se hacía lo siguiente: ya sabemos que en UNIVERSAL vamos a tener los datos limpios, ahora podríamos construir un proceso que se conecta a esas tablas de datos ya limpias y las empieza a procesar y por ejemplo aquí salga el reporte o salga el modelo de red neuronal o lo que estemos implementando. Pero ¿qué es lo que pasa? como yo les dije, las reglas de calidad no necesariamente son las mismas para todas las soluciones que vamos a implementar, por ejemplo, a un proceso de negocio si le importa que la edad sea mayor a cero y otro proceso le da exactamente igual, porque, se enfoca en procesar las estaturas de las personas. Así que, aquí en la capa UNIVERSAL van reglas de calidad genéricas, por ejemplo, no pueden haber registros con todos sus valores nulos, eso se tiene que cumplir para cualquier tabla y en la capa SMART la propia solución ya define sus propias reglas de calidad particulares, por ejemplo, en esta solución de la capa SMART la persona debe tener edades mayores a cero. Digamos que otra solución que hace otra cosa con la tabla limpia de la capa UNIVERSAL, la edad le da exactamente igual, puede tener edad menor a cero, no le interesa, pero las estaturas tienen que ser mayores a cero y tiene que tener registrado un correo electrónico en el campo de correo porque le quiere enviar un correo. Entonces, cada una de las soluciones define sus propias reglas de calidad. **El output de una tabla con reglas de calidad particulares (en la capa SMART) va una pequeña base de datos conocida como `DATA MESH`**. El **`DATA MESH` lo que tiene son las resultantes de reglas de calidad particulares a cada solución** y ahora sí construimos la solución, quizá quiere hacer un proceso de reportería, quizá quiere hacer un modelo de red neuronal.

##### **SMART**

La zona de SMART se conecta a los datos que ya están limpios y ya hace su proceso de ETL, su proceso de reporting, su red neuronal o lo que quiera hacer. Esta es la capa de soluciones (SMART), EN DONDE TU YA TE PONES A IMPLEMENTAR TODO LO QUE QUIERAS CODIFICAR, DESDE UNA REPORTERÍA HASTA UN MODELO DE RED NEURONAL. **Dentro de esa capa de soluciones, cada solución define su propio `DATA MESH`**, **QUE EN ESENCIA SON REGLAS PARTICULARES, QUE CADA SOLUCIÓN NECESITA, QUE NO PUEDE SER REALIZADA EN LA CAPA UNIVERSAL, porque, se contradecirían entre sí**. Así que cada solución que defina sus propias reglas de calidad, qué es lo que entiende por una tabla 'Persona' que esté lista para ser procesada. Esas resultantes van en una pequeña base de datos llamada DATA MESH, el cual, ya es el punto de partida para poder hacer la solución que tú necesitas. 

##### **DIAGRAMA**

<center><img src="https://i.postimg.cc/xdDCDB6s/p56.png"></center>
<center><img src="https://i.postimg.cc/W4v2fg4C/p57.png"></center>

#### **`CAPA ETL`**

##### **ETL PARA DATOS ESTRUCTURADOS**


Entonces a nivel conceptual hay que aterrizarlo a un mayor nivel de detalle para cada tipo de dato que vamos a procesar. Por 
ejemplo, para el procesamiento de datos estructurados, vamos a poner los ejemplos en este momento solamente con procesos batch, 
pero, en el real time es exactamente lo mismo, solo que van a cambiar las tecnologías, así que, por ahora simplemente hablemos 
conceptualmente. En el procesamiento de los datos estructurados para la ingesta, tenemos esta estrategia, habíamos llegado 
hasta el Gateway, exportamos los datos de la Fuente de datos a un Fileserver, los movemos dentro de un Gateway, que puede ser 
un directorio compartido de este Fileserver con el Gateway y subimos el archivo de datos. ¿Donde se sube? a la zona de 
LANDING_TMP. Cualquier cosa que entre al entorno del Data lake, tiene que ir esta zona de LANDING_TMP. Son generalmente 
archivos de texto plano, porque, los están exportando en CSV con algún separador. ¿Dónde van a vivir esos archivos que vamos 
a ir importando? van a vivir en tablas, porque, son archivos estructurados, entonces, si es data estructurada hay que mejor 
ver el archivo como una tabla. Esto tecnológicamente hablando se hace con HIVE. Una vez que hemos ingestado el archivo, ¿por 
qué se llama LANDING_TMP a esta capa? porque su único objetivo es hacer el CTRL + C, capturar el archivo que el día de hoy nos 
están dejando. Ahora en la capa LANDING vamos a mover ese archivo a la partición correspondiente, si estamos capturando la 
tabla 'Persona' debe de existir tanto en LANDING_TMP como en LANDING, solo que en LANDING va a estar particionada por fecha y 
en una partición estará el archivo del día de ayer, en otra partición el de que se dio anteayer y el día de hoy se creó una 
nueva partición y ahí se escribe el archivo de datos y de esa manera aquí es donde ya viven los datos acumulados. LANDING_TMP 
solamente es una zona de paso temporal. Adicionalmente a eso al mover el archivo de texto plano y colocarlo en la partición 
correspondiente del día de hoy, nunca hay que colocarlo en texto plano, porque, ya vamos a empezar a procesar esos datos, el 
siguiente paso es, una vez que se capture la data, ahora hay que limpiar los datos, las edades tienen que ser mayores a cero. 
Para hacer procesamiento de datos, los motores de Big data podrían hacerlo en texto plano, pero, esos archivos son 
extremadamente lentos, existen binaria archivos binarios de rápido procesamiento como **`AVRO`**, **`ORC`** y **`PARQUET`**. Acá los vamos a binarizar en AVRO. La binarización lo único que hace es convertir el archivo de texto plano en un archivo binario que si tu lo abres con un editor de texto vas a ver código binario, ya no vas a ver información. Pero si lo abres a nivel de una tabla vas a ver datos, si esto tenía 100 registros, pues, vas a ver también los 100 registros ahí. Como son datos estructurados los vamos a manejar a nivel de tabla, no a nivel de archivo, así que mejor el que el archivo esté binarizado para que pueda procesarse 
rápido. Adicionalmente a eso, vamos a comprimir también de los datos. La ventaja de hacer una binarización no solamente nos 
ayuda a aumentar la velocidad en las consultas, sino, que también nos ayuda a ahorrar espacio en disco duro. El formato de 
compresión estándar que existe el día de hoy en entornos de Big data se llama **`SNAPPY`**, que permite paralelizar los códigos, a 
pesar de que la data esté comprimida. SNAPPY es el estándar de facto en compresión para los datos, así que al momento de 
binarizar también vamos a comprimir. ¿En cuánto puede llegar a comprimir SNAPPY? en el mejor de los casos puede comprimir los 
datos hasta en un 80% y recuerda que estamos en un entorno de Big data, si estamos en gestando una tabla de datos de 1 TB, al 
comprimirlo con SNAPPY, ese terabyte que son 1.000 GB de datos, podemos reducirlo a 200 GB de datos y nos estamos ahorrando 
800 GB, eso es bastante información. Por supuesto que la compresión tiene una penalización, las consultas van a correr un 
poquito más lentas de que si solo estuvieran en AVRO, pero miren la ganancia y ahorro de disco duro. El estándar en el caso de 
procesos batch, la data que sea batchera comprímela, pero la data que sea en tiempo real, esa sí no la vamos a comprimir, 
porque, el proceso se va a hacer más lento si lo comprimimos, entonces, ahí no se comprime. Entonces, en la capa LANDING se 
binariza, se comprime y ese archivo binarizado y comprimido se guarda en la partición correspondiente. 

<center><img src="https://i.postimg.cc/gjnd6KMX/p58.png"></center>


##### **ETL PARA DATOS SEMI ESTRUCTURADOS**

Ahora en el caso de los archivos semiestructurados, la estrategia es la misma. ¿Cuál es la diferencia? los archivos 
probablemente ya no vengan de una base de datos, van a venir de servidores en donde habrá archivos JSON y XML y los tendremos 
que colocar dentro del LANDING_TMP. Pero ya no van a vivir en tablas, porque, son archivos semi estructurados, no los podemos 
ver como tablas. Dentro de LANDING_TMP, digamos que estamos ingestando el archivo 'Persona', pues, tendremos que crear un 
directorio 'Persona' y dentro colocar ese archivo JSON. Si son archivos semiestructurados, eso se maneja a nivel de directorios, 
debemos de crear un directorio para cada entidad y dentro guardar el archivo. Ahora para moverlo hacia LANDING, ahí vamos a 
repetir el mismo proceso, lo vamos a binarizar y comprimir, esos datos semiestructurados sean JSON o XML pueden ser binarizados 
en AVRO, sólo que ahora la estrategia de particiones se va a manejar el directorio 'Persona' y en subdirectorios se irán 
agregando los archivos según fecha. Estos archivos JSON y XML también estarán binarizados en AVRO, ya que, AVRO soporta 
formatos estructurados y también semiestructurados y por supuesto, también tendrán que estar comprimidos. Así es como iremos 
poblando la capa LANDING. 
<center><img src="https://i.postimg.cc/8k0Tgq5h/p59.png"></center>

##### **ETL PARA DATOS NO ESTRUCTURADOS**

Ahora, en el caso de los archivos no estructurados, aquí es un poquito diferente. En esencia, la forma en cómo vamos a mover 
los archivos de las fuentes de datos es la misma que el semiestructurado, pero, ya no los vamos a binarizar. La fuente de datos 
tiene archivos de imágenes o videos o audios, da igual el formato de archivo, todo tiene que ir al Gateway. ¿Cómo lo subimos al 
LANDING_TMP? pues es simplemente un directorio, en donde subiremos los archivos de imágenes de ese día, por ejemplo, digamos 
que tenemos grabaciones de una cámara de seguridad y se generaron un video por cada hora de esa cámara de seguridad, entonces, 
tendríamos que subir 24 archivos de vídeo dentro de este directorio y este directorio llevaría el nombre "Grabación de la 
cámara de seguridad 27”, por darle un nombre. Y luego, lo único que tenemos que hacer es, una vez que los datos se ingresaron 
al LANDING_TMP, luego solamente los vamos a mover a la capa LANDING a su partición correspondiente, colocaremos las imágenes 
del día de anteayer, las del día de ayer y las del día de hoy y las del día de mañana. En LANDING ya no vamos a binarizar en 
algún formato de rápido procesamiento como lo hacíamos anteriormente, acá simplemente lo movemos a una partición y de esa 
manera vamos ordenando las capturas de los datos. 

Y listo, con eso ya hemos resuelto el primer problema. Ya sabemos para cualquier nivel de estructura cómo vamos a guardar los 
datos que vamos a capturar dentro del DATA LAKE. Hemos llegado a las capas de LANDING_TMP y LANDING. Así que sea cual sea el 
dato y el nivel de estructura de ese dato, ya sabemos, cómo va a aterrizar en LANDING, lo que es data estructurada y 
semiestructurada se va a binarizar en AVRO y se va a comprimir y lo que es datos no estructurados, bueno pues, simplemente va 
a vivir en las particiones y vamos a ir ordenando lo que estamos ingestando. 

<center><img src="https://i.postimg.cc/0jFvGJ3t/p60.png"></center>

#### **`CAPA MODELAMIENTO`**

##### **INTRODUCCIÓN**

Ahora, diariamente ya sabemos que en la capa LANDING los datos se van a ir acumulando y es donde va a vivir toda la 
información. Ahora, ya tenemos la estrategia de captura de datos, vamos a la parte de UNIVERSAL, en donde vamos a hacer el 
modelamiento de los datos. Aquí hay que tener bastante cuidado. ¿Qué significa modelar los datos? cuando hablamos de datos 
estructurados, el modelamiento de datos hace referencia a básicamente a 3 cosas: 

● **Seleccionar ciertos campos** de la tabla 'Persona', que, por ejemplo, vive en la zona de LANDING, castear los tipos de datos 
  correctos. Digamos que la tabla 'Persona’ tiene el nombre de la persona, su identificación, su salario, su edad, su sexo, su 
  estatura y muchos otros datos. Ahora, hay un rol conocido en las empresas como modelador de datos que define:  " … mira 
  nosotros somos un banco y a nosotros no nos interesan las estaturas de las personas, nos interesa su nombre, su edad, su 
  salario, su estado civil … ", digamos que estos datos. Entonces el modelamiento de datos implica que de los datos que estamos 
  ingestando no vamos a requerir todos los campos, eso depende de cada negocio, vamos a seleccionar los que realmente va a 
  utilizar negocio. 

● **Luego hay que CASTEAR a los tipos de datos correctos**, porque por ejemplo, puede ser que originalmente en la fuente de datos 
  que hemos ingestado la edad venga como cadena de caracteres, pero eso tiene que ser un número, hay que darle el tipo de dato 
  correcto, eso es castear, definir el esquema ya del dato, quiénes son número, quién es un entero, quién es un decimal, quién 
  es una cadena de caracteres. 

● **Una vez que ya los tenemos como tipos de datos correctos, ahí le aplicamos las reglas de calidad y ahí viene lo que les 
  expliqué, los que cumplan las reglas de calidad se van a una tabla y los que no se van una tabla reyectada**. En esencia, a 
  eso nos referimos con modelar, tener los datos listos y dejarlos en la capa UNIVERSAL para que las soluciones los puedan 
  consumir y ya procesarlos. Pero no es tan simple. En el caso de los datos estructurados, lo que les ha explicado es la forma 
  en cómo se trabaja, en el caso de los datos semiestructurados el modelamiento es diferente. **Dentro de `UNIVERSAL` lo que 
  debemos tener son `TABLAS ESTRUCTURADAS`. Acá por ejemplo van a venir `ARCHIVOS SEMIESTRUCTURADOS`, de alguna manera que aún
  no conocemos esos `ARCHIVOS SEMIESTRUCTURADOS` los vamos a convertir en `TABLAS ESTRUCTURADAS`. Lo mismo con los `ARCHIVOS NO 
  ESTRUCTURADOS`, de alguna manera que no conocemos vamos a tener que convertir esos archivos a `FORMATOS ESTRUCTURADOS`.** De 
  hecho, en algunos casos va a ser imposible colocarlos en tablas, pero sí lo vamos a manejar a nivel de archivos 
  estructurados, así que, EL OBJETIVO DE UNIVERSAL ES TENER LA DATA ESTRUCTURADA, LISTA PARA QUE SEA PROCESADA, ESTRUCTURADA Y 
  CON LAS REGLAS DE CALIDAD. 

<center><img src="https://i.postimg.cc/bNSpy3D4/p61.png"></center>
<center><img src="https://i.postimg.cc/NjhY5mRF/p62.png"></center>

##### **REGLAS DE CALIDAD**

Paralelamente, se tiene que también definir las reglas de calidad. En la arquitectura eso se le conoce como la VISTA DE 
CALIDAD. Eso no necesariamente lo hacen los modeladores, eso lo puede hacer el equipo de calidad que es un equipo de personas 
completamente diferentes, que conocen ya la naturaleza de las tablas, de las fuentes de datos. Por ejemplo, podrían decir lo 
siguiente: " … el modelador ¿qué es lo que definió para la  tabla ‘Persona’? estos campos y estos tipos de datos, … " bien, 
pero a nivel de procesamiento de datos, ¿qué significa datos limpios en esta tabla? por ejemplo, vamos a definir cuatro reglas 
de calidad, los identificadores de los registros no pueden ser nulos, el campo sexo solamente puede tomar 3 valores: masculino, 
femenino y nulo. El salario tiene que ser un número mayor a cero y también el salario tiene que ser menor a 100.000 USD y 
listo esas son las reglas de calidad que el equipo de calidad definió. Misma historia, esto no lo hace el arquitecto, el 
arquitecto solamente se encarga de que haya coherencia entre las definiciones. Esto lo hace el equipo de calidad. 

<center><img src="https://i.postimg.cc/6pdKRqh0/p63.png"></center>

##### **MODELAMIENTO DATOS ESTRUCTURADOS**

La responsabilidad que nosotros tenemos como arquitectos es integrar el modelo definido por los modeladores y las reglas de 
calidad, dentro de la arquitectura y ¿cómo se hace? ahora sí vemos el patrón de diseño, ya sabemos que, lo que hemos ingestado 
desde la fuente de datos (la tabla 'Persona' que vive en la zona LANDING y está binarizado en AVRO) en función de lo que el 
modelador haya definido, seleccionaremos de esa tabla algunos campos y lo castearemos a los tipos de datos correctos y en 
función de lo que el equipo de calidad haya definido, en otra tercera cajita debemos de implementar esas reglas de calidad. 
Aquellos registros que cumplan las reglas de calidad aterrizarán en una tabla que se llamará exactamente igual que la otra 
tabla, pero, que vivirá en la zona de UNIVERSAL. Adicionalmente, a eso ¿en las fuentes de datos qué es lo que pasa en la vida
real? por ejemplo, en la fuente de datos el día de hoy, digamos que estamos conectándonos a la tabla 'Persona' desde la base 
de datos Oracle y esta tiene 10 campos, pero, por necesidad de negocio en la fuente de datos ahora le agregan un campo 11. La 
ventaja de tener binarizada la información en AVRO, es que nos permiten guardar esa evolución de las estructuras de datos 
dentro de la tabla, cada archivo va a guardar el formato de estructura de tablas con la que fue ingestado el día de hoy, por 
ejemplo, durante 10 años hemos estado ingestando datos que tienen 10 campos y el día de mañana la fuente de datos cambia a 11 
campos, pues, ese nuevo archivo binarizado en AVRO va a incluir dentro del archivo en una zona de metadatos ese nuevo campo 
número 11. Así que esa es la ventaja de AVRO permite tener esquemas flexibles. **Pero notemos que en la capa `UNIVERSAL` ya tenemos esquemas bien definidos por los moderadores, entonces, existe un formato binarizado llamado `PARQUET`, que es más rígido, no es tan flexible como `AVRO`, es ideal para hacer estas implementaciones en donde ya sabemos exactamente qué campos necesitamos. De hecho el `PARQUET` es mucho más rápido que el formato `AVRO`, porque, ya es un esquema fijo, por eso la capa `UNIVERSAL` se recomienda que esté en `PARQUET`.** En el caso de que los registros no cumplan las reglas de calidad, entonces, esos registros 
deberán de ser enviados a otra tabla que tendrá el mismo nombre de la fuente de datos, pero con un sufijo de "reject” que 
serán reyectados. Adicionalmente a eso, esta tabla 'Persona reject' va a tener los mismos campos que la tabla 'Persona', pero 
sus tipos de datos van a ser siempre cadenas de caracteres. ¿Por qué? para poder guardar cualquier cosa que se reyecte. Qué 
tal, por ejemplo, si alguien en la edad o en el campo salario se equivocaron y colocaron el nombre de la persona, se reyecta. 
Para poder guardar ese error eso tiene que ser un string, si no, no se podría guardar ese error. Por eso es que esta tabla de 
reyectados tiene que ser siempre de cadena de caracteres. Esto es para el caso de los datos estructurados. 

<center><img src="https://i.postimg.cc/R04BTmsz/p64.png"></center>

##### **MODELAMIENTO DE DATOS SEMI ESTRUCTURADOS**

**Ahora y ¿qué pasa para los datos semiestructurados? el objetivo con ellos va a ser estructurar los datos**. La buena noticia 
aquí es que da igual si son archivos JSON, XML, BSON o cualquier otro tipo de semiestructuras. Las reglas van a ser las mismas. 
Voy a ponerlo de manera muy genérica el día de hoy y ya cuando lo veamos en código vamos a verlo en vivo, primero lo 
entendamoslo conceptualmente. Acá, por ejemplo, cada registro de este archivo JSON tiene datos de Persona, Empresa y datos de 
Transacción. Entonces, podríamos crear 3 tablas la tabla 'Persona', 'Empresa' y 'Transacción'. Colocaremos los datos en 
registros de la Persona en una tabla, los de la Empresa en otra tabla y los datos propios de la Transacción en esa tercera 
tabla. Y de esa manera un archivo JSON se convirtió en 3 tablas estructuradas. Una vez que tenemos esas tablas estructuradas, 
lo siguiente es aplicar lo que previamente explicamos: cada una de ellas tiene que seleccionar ciertos campos, castearlos y 
aplicarlas las reglas de calidad y listo, ya tenemos esas tablas estructuradas. **También es muy importante entender que estas 
resultantes finales viven en UNIVERSAL, pero, se apoyan en tablas temporales para poder hacer estos procesos de seleccionar, 
castear y aplicar reglas de calidad, así que generalmente hay una zona del DATA LAKE en donde escribimos estas tablas 
temporales que van a vivir solamente para procesar, termina el proceso y como ya tenemos las tablas resultantes finales listas, 
se eliminan para ahorrar espacio en disco duro,** eso también es parte del patrón de diseño, solo las resultantes finales tienen 
que ser permanentes, las tablas temporales se eliminan una vez finalizado el proceso. Así que conceptualmente, no hay mucha 
complicación al momento de definir cómo se modelan los datos semiestructurados, el truco es esta parte de "Estructurar los 
datos”. De alguna manera tratar de estructurar los campos en diferentes tablas y luego ya tratarlos como tablas estructuradas. 

<center><img src="https://i.postimg.cc/MG58PLNf/p65.png"></center>

##### **MODELAMIENTO DE DATOS NO ESTRUCTURADOS**

<center><img src="https://i.postimg.cc/fLbQ6KQ2/p66.png"></center>
<center><img src="https://i.postimg.cc/XJD63CVV/p67.png"></center>
<center><img src="https://i.postimg.cc/ZnGkvzB2/p68.png"></center>
<center><img src="https://i.postimg.cc/L5YKs8Xj/p69.png"></center>
<center><img src="https://i.postimg.cc/J7vC2Z6x/p70.png"></center>
<center><img src="https://i.postimg.cc/5NzWdyDV/p71.png"></center>
<center><img src="https://i.postimg.cc/JzWV1mWB/p72.png"></center>

### **Spark**

#### **Encadenamiento de procesos (dataframes)**

##### **Dataframe de ejemplo**

```scala
//Objetos para definir la metadata
import org.apache.spark.sql.types.{StructType, StructField}

//Importamos los tipos de datos que usaremos
import org.apache.spark.sql.types.{StringType, IntegerType, DoubleType}

//Podemos importar todos los utilitarios con la siguiente sentencia
import org.apache.spark.sql.types._

//Importamos todos los objetos utilitarios dentro de una variable
import org.apache.spark.sql.{functions => f}


// DBTITLE 1,2. Lectura de datos
//Leemos el archivo de persona
var dfPersona = spark.read.format("csv").option("header", "true").option("delimiter", "|").schema(
    StructType(
        Array(
            StructField("ID", StringType, true),
            StructField("NOMBRE", StringType, true),
            StructField("TELEFONO", StringType, true),
            StructField("CORREO", StringType, true),
            StructField("FECHA_INGRESO", StringType, true),
            StructField("EDAD", IntegerType, true),
            StructField("SALARIO", DoubleType, true),
            StructField("ID_EMPRESA", StringType, true)
        )
    )
).load("dbfs:///FileStore/_spark/persona.data")

//Mostramos los datos
dfPersona.show(truncate=false)

dfPersona:org.apache.spark.sql.DataFrame
ID:string
NOMBRE:string
TELEFONO:string
CORREO:string
FECHA_INGRESO:string
EDAD:integer
SALARIO:double
ID_EMPRESA:string

+---+---------+--------------+---------------------------------------+-------------+----+-------+----------+
|ID |NOMBRE   |TELEFONO      |CORREO                                 |FECHA_INGRESO|EDAD|SALARIO|ID_EMPRESA|
+---+---------+--------------+---------------------------------------+-------------+----+-------+----------+
|1  |Carl     |1-745-633-9145|arcu.Sed.et@ante.co.uk                 |2004-04-23   |32  |20095.0|5         |
|2  |Priscilla|155-2498      |Donec.egestas.Aliquam@volutpatnunc.edu |2019-02-17   |34  |9298.0 |2         |
|3  |Jocelyn  |1-204-956-8594|amet.diam@lobortis.co.uk               |2002-08-01   |27  |10853.0|3         |
|4  |Aidan    |1-719-862-9385|euismod.et.commodo@nibhlaciniaorci.edu |2018-11-06   |29  |3387.0 |10        |
|5  |Leandra  |839-8044      |at@pretiumetrutrum.com                 |2002-10-10   |41  |22102.0|1         |
|6  |Bert     |797-4453      |a.felis.ullamcorper@arcu.org           |2017-04-25   |70  |7800.0 |7         |
|7  |Mark     |1-680-102-6792|Quisque.ac@placerat.ca                 |2006-04-21   |52  |8112.0 |5         |
|8  |Jonah    |214-2975      |eu.ultrices.sit@vitae.ca               |2017-10-07   |23  |17040.0|5         |
|9  |Hanae    |935-2277      |eu@Nunc.ca                             |2003-05-25   |69  |6834.0 |3         |
|10 |Cadman   |1-866-561-2701|orci.adipiscing.non@semperNam.ca       |2001-05-19   |19  |7996.0 |7         |
|11 |Melyssa  |596-7736      |vel@vulputateposuerevulputate.net      |2008-10-14   |48  |4913.0 |8         |
|12 |Tanner   |1-739-776-7897|arcu.Aliquam.ultrices@sociis.com       |2011-05-10   |24  |19943.0|8         |
|13 |Trevor   |512-1955      |Nunc.quis.arcu@egestasa.org            |2010-08-06   |34  |9501.0 |5         |
|14 |Allen    |733-2795      |felis.Donec@necleo.org                 |2005-03-07   |59  |16289.0|2         |
|15 |Wanda    |359-6973      |Nam.nulla.magna@In.org                 |2005-08-21   |27  |1539.0 |5         |
|16 |Alden    |341-8522      |odio@morbitristiquesenectus.ca         |2006-12-05   |26  |3377.0 |2         |
|17 |Omar     |720-1543      |Phasellus.vitae.mauris@sollicitudin.net|2014-06-24   |60  |6851.0 |6         |
|18 |Owen     |1-167-335-7541|sociis@erat.com                        |2002-04-09   |34  |4759.0 |7         |
|19 |Laura    |1-974-623-2057|mollis@ornare.ca                       |2017-03-09   |70  |17403.0|4         |
|20 |Emery    |1-672-840-0264|at.nisi@vel.org                        |2004-02-27   |24  |18752.0|9         |
+---+---------+--------------+---------------------------------------+-------------+----+-------+----------+
only showing top 20 rows
```

##### **Forma 1**

- Al ejecutar la siguiente sección de código, solamente hemos definido cómo se crearán los dataframes "df1", "df2", "df3" y "dfResultado"
- Los dataframes aún no se han creado y no ocupan memoria RAM en el clúster
- Para que se cree, deberemos llamar a un action
- Generalmente con los dataframes queremos hacer dos cosas, verlos o almacenarlos
- Estos son "actions" que ejecutan la cadena de procesos
- Al ejecutar "dfResultado.show()", la cadena de procesos asociada se ejecuta y se cargan en RAM todos los dataframes
- Si volvemos a llamar al "action", nuevamente se tendrá que recalcular toda la cadena de procesos
- Si la cadena de procesos se demora 1 hora, para que el action "show" muestre los datos, cada vez que lo ejecutemos deberemos esperar 1 hora

```scala
//DEFINIMOS LOS PASOS DE PROCESAMIENTO

//PASO 1: Agrupamos los datos según la edad
var df1 = dfPersona.groupBy(dfPersona.col("EDAD")).agg(
	f.count(dfPersona.col("EDAD")).alias("CANTIDAD"), 
	f.min(dfPersona.col("FECHA_INGRESO")).alias("FECHA_CONTRATO_MAS_RECIENTE"), 
	f.sum(dfPersona.col("SALARIO")).alias("SUMA_SALARIOS"), 
	f.max(dfPersona.col("SALARIO")).alias("SALARIO_MAYOR")
)

df1:org.apache.spark.sql.DataFrame
EDAD:integer
CANTIDAD:long
FECHA_CONTRATO_MAS_RECIENTE:string
SUMA_SALARIOS:double
SALARIO_MAYOR:double

//PASO 2: Filtramos por una EDAD
var df2 = df1.filter(df1.col("EDAD") > 35)

df2:org.apache.spark.sql.Dataset[org.apache.spark.sql.Row]
EDAD:integer
CANTIDAD:long
FECHA_CONTRATO_MAS_RECIENTE:string
SUMA_SALARIOS:double
SALARIO_MAYOR:double

//PASO 3: Filtramos por SUMA_SALARIOS
var df3 = df2.filter(df2.col("SUMA_SALARIOS") > 20000)

df3:org.apache.spark.sql.Dataset[org.apache.spark.sql.Row]
EDAD:integer
CANTIDAD:long
FECHA_CONTRATO_MAS_RECIENTE:string
SUMA_SALARIOS:double
SALARIO_MAYOR:double

//PASO 4: Filtramos por SALARIO_MAYOR
var dfResultado = df3.filter(df3.col("SALARIO_MAYOR") > 1000)

dfResultado:org.apache.spark.sql.Dataset[org.apache.spark.sql.Row]
EDAD:integer
CANTIDAD:long
FECHA_CONTRATO_MAS_RECIENTE:string
SUMA_SALARIOS:double
SALARIO_MAYOR:double

//Al ejecutar "dfResultado.show()", la cadena de procesos asociada se ejecuta y se cargan en RAM todos los dataframes
dfResultado.show()

+----+--------+---------------------------+-------------+-------------+
|EDAD|CANTIDAD|FECHA_CONTRATO_MAS_RECIENTE|SUMA_SALARIOS|SALARIO_MAYOR|
+----+--------+---------------------------+-------------+-------------+
|  41|       4|                 2002-10-10|      48309.0|      22102.0|
|  42|       5|                 2002-05-02|      62417.0|      22038.0|
|  46|       2|                 2000-04-03|      32820.0|      22953.0|
|  47|       2|                 2001-09-17|      35036.0|      21591.0|
|  48|       2|                 2008-10-14|      29218.0|      24305.0|
|  52|       3|                 2006-04-21|      32373.0|      14756.0|
|  55|       2|                 2000-08-16|      23923.0|      13813.0|
|  58|       3|                 2002-05-31|      45195.0|      23975.0|
|  61|       1|                 2008-03-24|      21452.0|      21452.0|
|  64|       2|                 2011-10-19|      41851.0|      22838.0|
|  67|       3|                 2002-08-21|      45357.0|      24575.0|
|  70|       3|                 2012-04-05|      36315.0|      17403.0|
+----+--------+---------------------------+-------------+-------------+


//Si volvemos a llamar al "action", nuevamente se tendrá que recalcular toda la cadena de procesos
//Si la cadena de procesos se demora 1 hora, para que el action "show" muestre los datos, cada vez que lo ejecutemos deberemos esperar 1 hora
dfResultado.show()

+----+--------+---------------------------+-------------+-------------+
|EDAD|CANTIDAD|FECHA_CONTRATO_MAS_RECIENTE|SUMA_SALARIOS|SALARIO_MAYOR|
+----+--------+---------------------------+-------------+-------------+
|  41|       4|                 2002-10-10|      48309.0|      22102.0|
|  42|       5|                 2002-05-02|      62417.0|      22038.0|
|  46|       2|                 2000-04-03|      32820.0|      22953.0|
|  47|       2|                 2001-09-17|      35036.0|      21591.0|
|  48|       2|                 2008-10-14|      29218.0|      24305.0|
|  52|       3|                 2006-04-21|      32373.0|      14756.0|
|  55|       2|                 2000-08-16|      23923.0|      13813.0|
|  58|       3|                 2002-05-31|      45195.0|      23975.0|
|  61|       1|                 2008-03-24|      21452.0|      21452.0|
|  64|       2|                 2011-10-19|      41851.0|      22838.0|
|  67|       3|                 2002-08-21|      45357.0|      24575.0|
|  70|       3|                 2012-04-05|      36315.0|      17403.0|
+----+--------+---------------------------+-------------+-------------+
```

##### **Forma 2**

- Al ejecutar la siguiente sección de código, solamente hemos definido cómo se creará el dataframe "dfResultado2"
- Cada paso, también crea en memoria RAM un dataframe ("P1", "P2", "P3"), sólo que ya no lo tenemos en una variable
- Eso significa que encadenar procesos utiliza la misma cantidad de memoria RAM que el hacerlo en variables separadas
- Los dataframes aún no se han creado y no ocupan memoria RAM en el clúster
- Para que se cree, deberemos llamar a un action
- Al ejecutar "dfResultado2.show()", la cadena de procesos asociada se ejecuta y se cargan en RAM todos los dataframes

```scala
//Definimos los pasos de procesamiento en una única cadena de proceso
var dfResultado2 = dfPersona.groupBy(dfPersona.col("EDAD")).agg(
	f.count(dfPersona.col("EDAD")).alias("CANTIDAD"), 
	f.min(dfPersona.col("FECHA_INGRESO")).alias("FECHA_CONTRATO_MAS_RECIENTE"), 
	f.sum(dfPersona.col("SALARIO")).alias("SUMA_SALARIOS"), 
	f.max(dfPersona.col("SALARIO")).alias("SALARIO_MAYOR")
).alias("P1").
filter(f.col("P1.EDAD") > 35).alias("P2").
filter(f.col("P2.SUMA_SALARIOS") > 20000).alias("P3").
filter(f.col("P3.SALARIO_MAYOR") > 1000)

dfResultado2:org.apache.spark.sql.Dataset[org.apache.spark.sql.Row]
EDAD:integer
CANTIDAD:long
FECHA_CONTRATO_MAS_RECIENTE:string
SUMA_SALARIOS:double
SALARIO_MAYOR:double

//Al ejecutar "dfResultado2.show()", la cadena de procesos asociada se ejecuta y se cargan en RAM todos los dataframes
dfResultado2.show()

+----+--------+---------------------------+-------------+-------------+
|EDAD|CANTIDAD|FECHA_CONTRATO_MAS_RECIENTE|SUMA_SALARIOS|SALARIO_MAYOR|
+----+--------+---------------------------+-------------+-------------+
|  41|       4|                 2002-10-10|      48309.0|      22102.0|
|  42|       5|                 2002-05-02|      62417.0|      22038.0|
|  46|       2|                 2000-04-03|      32820.0|      22953.0|
|  47|       2|                 2001-09-17|      35036.0|      21591.0|
|  48|       2|                 2008-10-14|      29218.0|      24305.0|
|  52|       3|                 2006-04-21|      32373.0|      14756.0|
|  55|       2|                 2000-08-16|      23923.0|      13813.0|
|  58|       3|                 2002-05-31|      45195.0|      23975.0|
|  61|       1|                 2008-03-24|      21452.0|      21452.0|
|  64|       2|                 2011-10-19|      41851.0|      22838.0|
|  67|       3|                 2002-08-21|      45357.0|      24575.0|
|  70|       3|                 2012-04-05|      36315.0|      17403.0|
+----+--------+---------------------------+-------------+-------------+
```

#### **`SparkSession`**

En Spark, la creación de una sesión (`SparkSession`) es esencial para interactuar con el entorno de Spark y ejecutar operaciones en un clúster de Spark. Aquí te explico cuándo y por qué debes utilizar `SparkSession` en tus aplicaciones Spark:

##### ¿Qué es SparkSession?

`SparkSession` es la entrada principal para interactuar con Spark y es parte de la API estructurada de Spark desde la versión 2.x en adelante. Reemplaza a las antiguas `SparkContext`, `SQLContext` y `HiveContext` que se usaban en versiones anteriores de Spark.

##### ¿Cuándo debes utilizar `SparkSession`?

1. **Iniciar una Aplicación Spark**: Siempre que desees escribir una aplicación que utilice Spark para procesar datos, necesitas crear una instancia de `SparkSession`. Esta instancia proporciona una conexión centralizada a Spark y permite configurar opciones específicas de la aplicación, como el nombre de la aplicación (`appName`) y configuraciones adicionales.

   ```python
   from pyspark.sql import SparkSession
   
   # Inicia la sesión de Spark
   spark = SparkSession.builder.appName("Mi Aplicación Spark").getOrCreate()
   ```

   - `appName`: Especifica un nombre descriptivo para tu aplicación Spark. Esto es útil para la monitorización y el seguimiento en el entorno de Spark.

2. **Acceder a Funcionalidades de Spark SQL y DataFrame**: `SparkSession` proporciona métodos para crear DataFrames, ejecutar consultas SQL sobre datos estructurados y realizar operaciones distribuidas usando Spark SQL. Esto incluye leer y escribir datos desde/hacia diferentes fuentes de datos (HDFS, bases de datos relacionales, formatos de archivos como Parquet, CSV, etc.).

   ```python
   # Ejemplo de creación de DataFrame a partir de una lista de tuplas
   data = [("Alice", 1), ("Bob", 2), ("Carol", 3)]
   df = spark.createDataFrame(data, ["name", "age"])
   
   # Ejecutar una consulta SQL sobre el DataFrame
   df.createOrReplaceTempView("people")
   result = spark.sql("SELECT * FROM people WHERE age > 1")
   ```

3. **Configuración y Administración de Recursos**: A través de `SparkSession`, puedes configurar propiedades como la cantidad de recursos del clúster que debe utilizar tu aplicación, la configuración de logging, entre otras configuraciones específicas de la aplicación.

##### ¿Por qué es importante `SparkSession`?

- **Entrada Única**: Proporciona una única entrada para interactuar con todas las características de Spark, incluyendo SQL, Streaming y otros componentes.
  
- **Contexto de Aplicación**: `SparkSession` mantiene el contexto de tu aplicación Spark, gestionando recursos y coordinando las operaciones distribuidas en el clúster.

- **Optimización Interna**: Internamente, `SparkSession` gestiona la configuración óptima del motor de ejecución de Spark y la optimización de consultas SQL, lo cual es crucial para el rendimiento de las aplicaciones Spark.

##### Conclusión

En resumen, sí, siempre que estés escribiendo una aplicación que utilice Spark para procesar datos, debes iniciar una sesión de Spark utilizando `SparkSession`. Esto te permitirá aprovechar todas las capacidades de Spark para el procesamiento distribuido de datos, consultas SQL, machine learning, procesamiento de streaming, entre otros.

#### **¿Se debe importar `SparkSession` al utilizar la consola `spark-shell` y `pyspark`?**

No es necesario importar explícitamente `SparkSession` al utilizar `spark-shell`, ya que `spark-shell` proporciona automáticamente una instancia de `SparkSession` bajo el nombre de `spark`. En el contexto de `spark-shell`, `spark` es una instancia predefinida de `SparkSession` que ya está configurada y lista para ser utilizada para interactuar con Spark.

Por lo tanto, cuando ejecutas `spark-shell`, puedes comenzar a escribir y ejecutar comandos de Spark directamente utilizando el objeto `spark`. Por ejemplo, puedes cargar datos, realizar transformaciones, ejecutar consultas SQL y más, todo utilizando el objeto `spark`.

Aquí hay un ejemplo básico de cómo se usaría `spark-shell` para comenzar a trabajar con Spark:

1. Inicia `spark-shell` desde tu terminal:
   ```
   $ spark-shell
   ```

2. Una vez que `spark-shell` se inicie, automáticamente tendrás acceso al objeto `spark`:
   ```
   scala> spark
   res0: org.apache.spark.sql.SparkSession = org.apache.spark.sql.SparkSession@6d35a16d
   ```

3. A partir de aquí, puedes usar `spark` para realizar operaciones de Spark. Por ejemplo, puedes cargar un archivo CSV y mostrar los primeros registros:
   ```scala
   scala> val df = spark.read.csv("ruta/al/archivo.csv")
   scala> df.show()
   ```

4. También puedes ejecutar consultas SQL utilizando `spark.sql`:
   ```scala
   scala> df.createOrReplaceTempView("tabla")
   scala> val resultado = spark.sql("SELECT * FROM tabla WHERE edad > 30")
   scala> resultado.show()
   ```

En resumen, al utilizar `spark-shell`, no necesitas preocuparte por importar `SparkSession` explícitamente, ya que `spark-shell` maneja esa inicialización automáticamente para ti. Esto simplifica significativamente la interacción inicial con Spark y te permite concentrarte directamente en escribir y ejecutar tu código de análisis de datos.

En la consola `pyspark`, al igual que en `spark-shell` para Scala, no es necesario importar explícitamente `SparkSession`. `pyspark` proporciona automáticamente una instancia de `SparkSession` bajo el nombre de `spark` que puedes usar directamente.

Aquí te muestro cómo funciona en la práctica:

1. Inicia `pyspark` desde tu terminal:
   ```
   $ pyspark
   ```

2. Una vez que `pyspark` se inicie, automáticamente tendrás acceso al objeto `spark`:
   ```python
   >>> spark
   <pyspark.sql.session.SparkSession object at 0x7f0e0d74a850>
   ```

3. Puedes comenzar a usar `spark` para realizar operaciones de Spark. Por ejemplo, cargar un archivo CSV y mostrar los primeros registros:
   ```python
   >>> df = spark.read.csv("ruta/al/archivo.csv")
   >>> df.show()
   ```

4. También puedes ejecutar consultas SQL utilizando `spark.sql`:
   ```python
   >>> df.createOrReplaceTempView("tabla")
   >>> resultado = spark.sql("SELECT * FROM tabla WHERE edad > 30")
   >>> resultado.show()
   ```

En este caso, `spark` es la instancia de `SparkSession` que ya existe en `spark-shell` y `pyspark`, y puedes utilizar sus métodos directamente para cargar datos, realizar transformaciones, persistir datos, ejecutar consultas SQL, y mucho más.

Al igual que en `spark-shell`, en `pyspark` no es necesario importar `SparkSession` explícitamente. El objeto `spark` ya está disponible y configurado para que puedas comenzar a trabajar directamente con él para realizar análisis de datos utilizando PySpark.

En resumen, tanto en `spark-shell` para Scala como en `pyspark` para Python, `SparkSession` se maneja automáticamente y está disponible como `spark` para facilitar la interacción con Spark y realizar operaciones de manera eficiente sin la necesidad de importaciones adicionales.

#### **¿Se utiliza `SparkSession` o `SparkContext`?**

En el contexto de Spark 2.x y versiones posteriores, se utiliza principalmente `SparkSession` en lugar de `SparkContext` cuando estás trabajando con DataFrames y SQL en Spark. Aquí te explico brevemente la diferencia y cuándo utilizar cada uno:

1. **SparkSession**: Es la entrada principal para interactuar con Spark SQL y trabajar con DataFrames en Spark. `SparkSession` combina las funcionalidades previamente separadas de `SQLContext` y `HiveContext` en versiones anteriores de Spark. Proporciona métodos para leer datos, crear DataFrames, ejecutar consultas SQL, y más. En `spark-shell`, una instancia de `SparkSession` llamada `spark` se crea automáticamente para ti.

   Ejemplo de uso en `spark-shell`:

   ```scala

   // Cargar datos desde un archivo CSV utilizando SparkSession (spark)

   val df = spark.read.csv("hdfs://ruta/al/archivo.csv")

   // Mostrar los primeros registros del DataFrame

   df.show()

   ```

2. **SparkContext**: Es la API principal de bajo nivel para crear y manipular RDDs (Resilient Distributed Datasets) en Spark. RDDs son la abstracción básica de datos en Spark y son útiles cuando necesitas controlar el manejo de la distribución de datos a nivel más bajo. Sin embargo, para trabajar con DataFrames y ejecutar consultas SQL, `SparkSession` y la API de SQL son generalmente más convenientes y eficientes.


   Ejemplo de uso de `SparkContext` (para RDDs):

   ```scala

   val conf = new SparkConf().setAppName("MiApp").setMaster("local")

   val sc = new SparkContext(conf)


   // Crear un RDD a partir de una colección

   val rdd = sc.parallelize(Seq(1, 2, 3, 4, 5))


   // Operaciones sobre RDDs

   val suma = rdd.reduce(_ + _)

   println(s"La suma es: $suma")


   // Detener SparkContext al finalizar

   sc.stop()

   ```


En resumen, para la mayoría de los casos de uso modernos que involucran DataFrames, SQL y manipulación de datos estructurados en Spark, utilizarás `SparkSession`. `SparkContext` sigue siendo relevante si necesitas trabajar directamente con RDDs o si estás trabajando en una versión anterior de Spark donde `SparkSession` no está disponible.


#### **`spark-submit`**

**Spark-submit** es un commando que nos permite ejecutar **Spark Applications** en un cluster. Es importante conocer algunas de sus opciones.

In [ ]:
# Sintáxis del comando
spark-submit --class --master --deploy-mode [application-args]

In [ ]:
# Aplicación Pyspark
spark-submit --master yarn \
             --deploy-mode cluster \
             --driver-cores 1 \
             --driver-memory 4G \
             --num-executors 4 \
             --supervise \
             --executor-cores 4 \
             --executor-memory 16G \
             --name nombre-aplicacion \ 
             hello-spark.py

In [ ]:
# Aplicacion Scala
spark-submit --class md.learning.HelloSpark \
             --master yarn \
             --deploy-mode cluster \
             --driver-cores 1 \
             --driver-memory 4G \
             --num-executors 4 \
             --supervise \
             --executor-cores 4 \
             --executor-memory 16G \
             hello-spark.jar

In [ ]:
# No disponible para PySpark, indica el nombre de la clase donde definiste el main
--class

In [ ]:
# Indica al cluster manager si usas yarn, mesos, kubernates, etc.
# --master yarn: Especifica que el gestor de recursos utilizado es YARN. YARN se encargará de administrar los recursos del clúster para la aplicación Spark.
# Si se desea correr en local, seteamos local[3] indica que tendra 3 hilos.
--master -> yarn, local[3]

In [ ]:
# El modo en que se desplegara, no existe local, este es una configuración.
# --deploy-mode -> cluster: Indica que el modo de despliegue de la aplicación es en modo cluster. Esto significa que el Spark Driver se ejecutará en un nodo del clúster administrado por YARN, no en el nodo desde donde se lanza spark-submit.
--deploy-mode -> client o cluster

In [ ]:
# Nos permite agregar configuraciones adicionales.
--conf -> spark.executor.memoryOverhead = 0.20

In [ ]:
# Cantidad de CPU que tendra nuestro driver container.
# Especifica la cantidad de núcleos de CPU que se asignarán al Spark Driver. En este caso, se asigna 1 núcleo de CPU al Spark Driver.
--driver-cores -> 2

In [ ]:
# Cantidad de RAM que tendra nuestro driver container.
# Especifica la cantidad de memoria RAM que se asignará al Spark Driver. Aquí se asignan 8 GB de memoria RAM.
--driver-memory -> 8G

In [ ]:
# Cantidad executor containers.
# Indica el número de ejecutores que se lanzarán en el clúster para ejecutar tareas Spark. En este caso, se lanzarán 4 ejecutores.
--num-executors -> 4

In [ ]:
# Indica que se debe supervisar la aplicación. Si la aplicación falla, Spark intentará reiniciarla automáticamente.
--supervise

In [ ]:
# Cantidad de CPU que tendra cada uno de nuestros executors containers.
# Especifica la cantidad de núcleos de CPU que se asignarán a cada ejecutor. Cada ejecutor tendrá 4 núcleos de CPU.
--executor-cores -> 4

In [ ]:
# Cantidad de RAM que tendra cada uno de nuestros executors containers.
# Especifica la cantidad de memoria RAM que se asignará a cada ejecutor. Cada ejecutor tendrá 16 GB de memoria RAM.
--executor-memory -> 16G

In [ ]:
# Asigna un nombre descriptivo a la aplicación Spark que se está ejecutando. Este nombre puede ser útil para identificar la aplicación en el ResourceManager de YARN o en el historial de aplicaciones.
--name nombre-aplicacion

In [ ]:
# Especifica el archivo Python (hello-spark.py) que contiene el código de la aplicación Spark que se va a ejecutar.
hello-spark.py

#### **Ejemplos de `spark-submit`**

Documentación: https://spark.apache.org/docs/latest/submitting-applications.html#master-urls

In [ ]:
# Ejecutar la aplicación localmente en 8 cores
./bin/spark-submit \
--class org.apache.spark.examples.SparkPi \
--master local[8]
100

In [ ]:
# Ejecutar la aplicación en un clúster Spark Standalone en modo de ejecución "cliente"
./bin/spark-submit \
--class org.apache.spark.examples.SparkPi \
# Tambien podria ser una IP: --class spark://207.184.161.138:7077 \ 
--master spark://spark-master:7077 \ 
--executor-memory 20G \
--total-executor-cores 100 \
/path/to/examples.jar \
1000

In [ ]:
# Ejecutar la aplicación en un clúster Spark Standalone en modo de ejecución "cluster" con "supervise"
./bin/spark-submit \
--class org.apache.spark.examples.SparkPi \
--master spark://spark-master:7077 \ 
--deploy-mode cluster \
--supervise \
--executor-memory 20G \
--total-executor-cores 100 \
/path/to/examples.jar \
1000

In [ ]:
# Run on a YARN cluster
./bin/spark-submit \
--class org.apache.spark.examples.SparkPi \
--master yarn \ 
--deploy-mode cluster \
--executor-memory 20G \
--num-executors 50 \
/path/to/examples.jar \
1000

In [ ]:
# Run a Python application on a Spark standalone cluster
./bin/spark-submit \
# Tambien podria ser una IP: --class spark://207.184.161.138:7077 \ 
--master spark://spark-master:7077 \ 
examples/src/main/python/pi.py \
1000

In [ ]:
# Run on a Mesos cluster in cluster deploy mode with supervise
./bin/spark-submit \
--class org.apache.spark.examples.SparkPi \
--master mesos://207.184.161.138:7077 \ 
--deploy-mode cluster \
--supervise \
--executor-memory 20G \
--total-executor-cores 100 \
http://path/to/examples.jar \
1000

In [ ]:
# Run on a Kubernetes cluster deploy mode
./bin/spark-submit \
--class org.apache.spark.examples.SparkPi \
--master k8s://xx.yy.zz.ww:443 \ 
--deploy-mode cluster \
--executor-memory 20G \
--num-executors 50 \
http://path/to/examples.jar \
1000

#### **Ejemplos de `SparkSession.builder`**

Documentación: https://spark.apache.org/docs/latest/configuration.html#loading-default-configurations

##### **Modos de ejecución en PySpark**

**Modo Local**: En este modo, Spark ejecuta todo en un único proceso localmente en tu máquina, sin utilizar el clúster de Hadoop. Es útil para desarrollo y pruebas en entornos locales donde no se necesita la escalabilidad del clúster Hadoop.

En este caso, `master("local[*]")` indica que estás ejecutando en modo local utilizando todos los núcleos disponibles en tu máquina.

   ```python
   from pyspark.sql import SparkSession
   spark = SparkSession.builder \
       .appName("MiApp") \
       .master("local[*]") \
       .getOrCreate()

   ```

**Modo Cliente**: En este modo, el programa driver (donde se inicia la aplicación) se ejecuta en el entorno desde el que se lanzó la aplicación (por ejemplo, el contenedor `spark-master`). Los recursos de cómputo para la ejecución de tareas se solicitan al clúster de Spark (por ejemplo, nodos `spark-worker` en tu caso). Este modo es típicamente utilizado para trabajar con grandes conjuntos de datos distribuidos y aprovechar la capacidad de procesamiento distribuido de Spark.

En este ejemplo, `master("spark://spark-master:7077")` especifica que estás conectándote al `spark-master` para ejecutar la aplicación en modo cliente.

   ```python
   from pyspark.sql import SparkSession
   spark = SparkSession.builder \
       .appName("MiApp") \
       .master("spark://spark-master:7077") \
       .getOrCreate()

   ```

**Modo Cluster**: Similar al modo cliente, pero en este caso el programa controlador se ejecuta dentro del clúster de Spark (no en el contenedor `spark-master`). Este modo es útil para aplicaciones que necesitan alta disponibilidad y tolerancia a fallos.

Aquí, además de especificar el `master`, configuramos `spark.submit.deployMode` como "cluster" para indicar que la aplicación se ejecutará en modo clúster.

   ```python
   from pyspark.sql import SparkSession
   spark = SparkSession.builder \
       .appName("MiApp") \
       .master("spark://spark-master:7077") \
       .config("spark.submit.deployMode", "cluster") \
       .getOrCreate()

   ```

#### **Indicación de una propiedad en el `spark-submit` y en el `SparkSession.builder`**

| NO ES NECESARIO INDICAR 2 VECES que el gestor de recursos va a ser yarn: spark = SparkSession.builder.appName("Nombre de mi aplicacion").`config("spark.master", "yarn")`.enableHiveSupport().getOrCreate() y luego en el spark-submit: spark-submit `--master yarn` |
|---|

Cuando configuras `SparkSession` en tu aplicación utilizando `SparkSession.builder`, especificas la configuración del gestor de recursos (`spark.master`) como `yarn`. Aquí está el ejemplo ajustado con esta configuración:

```python
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Nombre de mi aplicacion") \
    .config("spark.master", "yarn") \
    .enableHiveSupport() \
    .getOrCreate()
```

En este caso, `spark.master` se establece como `yarn`, lo que indica que quieres utilizar YARN como el gestor de recursos para ejecutar las aplicaciones Spark.

##### Uso de `spark-submit`

Al usar `spark-submit` para enviar tu aplicación Spark al clúster, generalmente especificas la configuración de `--master` y `--deploy-mode`. Sin embargo, si ya has configurado `spark.master` en tu aplicación Spark utilizando `SparkSession.builder`, no es necesario especificarlo nuevamente en `spark-submit`. 

Por lo tanto, en tu comando `spark-submit`, simplemente podrías incluir `--deploy-mode` y otros parámetros relevantes para el despliegue de la aplicación, como el nombre de la aplicación, recursos de driver y ejecutor, etc. Pero no necesitas especificar `--master yarn` nuevamente si ya lo has configurado en tu código de la aplicación.

Aquí te doy un ejemplo ajustado del comando `spark-submit` que refleja esto:

```bash
spark-submit --deploy-mode cluster \
             --driver-cores 1 \
             --driver-memory 4G \
             --num-executors 4 \
             --executor-cores 4 \
             --executor-memory 16G \
             --name nombre-aplicacion \
             hello-spark.py
```

##### Resumen

- En tu aplicación Python de Spark, configuras `SparkSession` para utilizar YARN como gestor de recursos mediante `.config("spark.master", "yarn")`.
- Cuando ejecutas `spark-submit`, puedes omitir `--master yarn` porque ya está configurado en tu aplicación Spark.
- Solo necesitas asegurarte de que `--deploy-mode` y otros parámetros relevantes estén configurados correctamente según las necesidades de tu aplicación y entorno de clúster.

Siguiendo estos pasos, tu aplicación Spark debería ejecutarse correctamente utilizando YARN como gestor de recursos, integrando la compatibilidad con Hive si has habilitado `enableHiveSupport()` en tu configuración de `SparkSession`.

#### **Utilizar tablas Hive desde Spark (`Compilar Hive con Spark`)**

Sí, es correcto que para habilitar el soporte de Hive en Spark utilizando `enableHiveSupport()` al crear una instancia de `SparkSession`, es necesario que Spark esté configurado correctamente para interactuar con Hive. Esto implica tener acceso al archivo de configuración `hive-site.xml`, que contiene la configuración necesaria para conectarse al metastore de Hive y utilizar las tablas y funciones definidas en Hive desde Spark.

Aquí te explico los pasos clave:

##### 1. Configuración de Hive en Spark

Para que Spark pueda interactuar con Hive, debes seguir estos pasos:

- **Asegúrate de tener Hive instalado**: Spark utiliza la biblioteca de Hive para la integración. Por lo tanto, es importante tener una instalación de Hive adecuada en el entorno donde Spark se está ejecutando.

- **Ubicación de hive-site.xml**: El archivo `hive-site.xml` contiene la configuración de conexión al metastore de Hive. Debes asegurarte de que este archivo esté disponible en la ruta correcta dentro del entorno de Spark.

- **Ruta del directorio `$SPARK_HOME/conf`**: Esta es la ubicación donde Spark busca los archivos de configuración, incluido `hive-site.xml`. Si `$SPARK_HOME` está correctamente definido en tu entorno, Spark automáticamente cargará la configuración de Hive de este archivo cuando sea necesario.

##### 2. Habilitación de Hive en Spark

Al crear una instancia de `SparkSession` en tu aplicación de Spark, debes habilitar el soporte de Hive utilizando `enableHiveSupport()`:

```python
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Mi aplicacion Spark") \
    .enableHiveSupport() \
    .getOrCreate()
```

##### 3. Copia de hive-site.xml

Si `$SPARK_HOME/conf` no contiene el archivo `hive-site.xml`, necesitarás asegurarte de copiarlo manualmente desde la configuración de Hive a esta ubicación. Esto garantiza que Spark pueda leer la configuración adecuada para conectarse al metastore de Hive y trabajar con las tablas definidas en Hive.

##### 4. Verificación

Para asegurarte de que todo esté configurado correctamente:

- Verifica que `hive-site.xml` esté presente en `$SPARK_HOME/conf`.
- Asegúrate de que la configuración dentro de `hive-site.xml` sea correcta para conectarte al metastore de Hive que deseas utilizar.
- Ejecuta tu aplicación de Spark y verifica que puedas acceder y manipular tablas de Hive desde Spark sin errores relacionados con la configuración de Hive.

En resumen, sí, para utilizar `enableHiveSupport()` en Spark, es necesario que Spark esté correctamente configurado con Hive, lo cual implica tener acceso a `hive-site.xml` en la ubicación adecuada (`$SPARK_HOME/conf`). Esto asegura una integración sin problemas entre Spark y Hive para el procesamiento de datos en ambos entornos.

#### **Utilizar `enableHiveSupport()`** 
- **spark = SparkSession.builder.appName("Mi aplicacion Spark").`enableHiveSupport()`.getOrCreate()**
- **spark = SparkSession.builder.appName("Mi aplicación Spark").getOrCreate()**

La elección entre los dos métodos de inicialización de `SparkSession` depende de si necesitas soporte para Hive en tu aplicación Spark o no. Aquí te explico la diferencia entre ambos:

##### Opción 1: `enableHiveSupport()`

```python
spark = SparkSession.builder.appName("Nombre de mi aplicacion").enableHiveSupport().getOrCreate()
```

- **Descripción**: Esta opción habilita el soporte de Hive en tu sesión de Spark. Hive es una infraestructura de data warehouse construida sobre Hadoop que proporciona un mecanismo para consultar y gestionar grandes conjuntos de datos almacenados en Hadoop HDFS. Cuando habilitas el soporte de Hive en `SparkSession`, puedes acceder a tablas Hive existentes en tu clúster y ejecutar consultas SQL sobre ellas utilizando Spark SQL.

- **Cuándo utilizarlo**: Utiliza esta opción si tu aplicación Spark necesita interactuar con tablas Hive existentes en tu entorno. Esto puede ser útil si tienes un sistema de almacenamiento de datos basado en Hive y quieres aprovechar las capacidades de procesamiento distribuido de Spark para consultar y manipular esos datos.

##### Opción 2: Sin `enableHiveSupport()`

```python
spark = SparkSession.builder.appName("Mi Aplicación Spark").getOrCreate()
```

- **Descripción**: Esta opción crea una sesión de Spark sin habilitar el soporte de Hive explícitamente. Aunque Spark puede interactuar con datos almacenados en HDFS u otros sistemas de archivos, no proporciona integración directa con las tablas de Hive ni con el metastore de Hive.

- **Cuándo utilizarlo**: Utiliza esta opción si tu aplicación Spark no necesita interactuar con tablas Hive específicas y simplemente procesa datos a partir de archivos, bases de datos u otros orígenes de datos compatibles con Spark. Esto es adecuado si estás desarrollando una aplicación Spark que no depende de Hive para almacenamiento ni para la ejecución de consultas SQL complejas sobre tablas Hive.

##### Consideraciones adicionales:

- **Overhead de inicialización**: Habilitar `enableHiveSupport()` puede añadir un pequeño overhead durante la inicialización de la sesión de Spark, ya que implica la configuración de conexiones con el metastore de Hive y la integración con el sistema de archivos de Hive.

- **Compatibilidad**: Si no estás seguro de si necesitas Hive en tu aplicación, es seguro comenzar sin habilitar `enableHiveSupport()`. Si más adelante necesitas interactuar con Hive, puedes modificar tu sesión de Spark para habilitar esta función sin problemas mayores.

En resumen, elige la opción que mejor se ajuste a las necesidades específicas de tu aplicación Spark: si necesitas acceso a tablas de Hive, elige la primera opción; si no necesitas integración con Hive, la segunda opción será más liviana y adecuada.

#### **Utilizar `config("spark.master", "<gestor de recursos>")`**: determinar cómo se gestionará la ejecución y la distribución de recursos

- **spark = SparkSession.builder.appName("Nombre de mi aplicacion").`config("spark.master", "local")`.enableHiveSupport().getOrCreate()**
- **spark = SparkSession.builder.appName("Nombre de mi aplicacion").`config("spark.master", "yarn")`.enableHiveSupport().getOrCreate()**
- **spark = SparkSession.builder.appName("Nombre de mi aplicacion").`config("spark.master", "mesos://host:port")`.enableHiveSupport().getOrCreate()**
- **spark = SparkSession.builder.appName("Nombre de mi aplicacion").`config("spark.master", "spark://host:port")`.enableHiveSupport().getOrCreate()**

Spark proporciona varias opciones para el parámetro `spark.master` que determinan cómo se gestionará la ejecución y la distribución de recursos:

1. **Modo Local sin Especificar el Número de Hilos**: Si simplemente utilizas `local` sin especificar el número de hilos, Spark ejecutará la aplicación en modo local utilizando tantos hilos como núcleos tenga tu máquina. Esto es útil cuando quieres que Spark utilice todos los recursos disponibles en tu máquina local.

   ```python
   .config("spark.master", "local")
   ```

2. **Modo Local con Especificación de Número de Hilos**: Puedes especificar el número exacto de hilos que Spark debe utilizar en modo local. Por ejemplo, `local[4]` indica que Spark utilizará 4 hilos para la ejecución.

   ```python
   .config("spark.master", "local[4]")
   ```

3. **Ejecución en un Clúster Utilizando YARN**: Si estás utilizando YARN como gestor de recursos, puedes especificar su dirección. Por ejemplo:

   ```python
   .config("spark.master", "yarn")
   ```

   Puedes ajustar esta configuración según el modo en que estés ejecutando YARN (por ejemplo, `yarn-client` o `yarn-cluster`).

4. **Ejecución Utilizando Mesos**: Si utilizas Mesos como gestor de recursos, puedes especificar su dirección:

   ```python
   .config("spark.master", "mesos://host:port")
   ```

   Donde `host:port` es la dirección del maestro de Mesos.

5. **Ejecución Utilizando Kubernetes**: Si estás utilizando Kubernetes como gestor de contenedores, puedes especificar su dirección:

   ```python
   .config("spark.master", "k8s://https://<k8s-apiserver-host>:<k8s-apiserver-port>")
   ```

   Donde `<k8s-apiserver-host>` y `<k8s-apiserver-port>` son la dirección y el puerto del servidor API de Kubernetes.

6. **Ejecución Utilizando Spark Standalone**: Si tienes un clúster Spark en modo standalone, puedes especificar la dirección de su maestro:

   ```python
   .config("spark.master", "spark://host:port")
   ```

   Donde `host:port` es la dirección y el puerto del maestro de Spark standalone.

##### Ejemplo de Uso General:

```python
from pyspark.sql import SparkSession

# Configura y crea la sesión de Spark para ejecución local con 4 hilos
spark = SparkSession.builder \
    .appName("Mi Aplicación Spark") \
    .config("spark.master", "local[4]") \
    .getOrCreate()

# A partir de este punto, puedes usar `spark` para trabajar con DataFrames, ejecutar consultas SQL, etc.
```

##### Consideraciones:

- **Ambiente de Ejecución**: La elección del modo de ejecución (`local`, `yarn`, `mesos`, `kubernetes`, `spark://`) depende del entorno en el que desees ejecutar tu aplicación Spark: desarrollo local, clúster de producción, o sistemas de gestión de contenedores como Kubernetes.

- **Escalabilidad y Recursos**: Ajusta el número de hilos o la configuración de tu entorno de clúster (`yarn`, `mesos`, `spark://`) según los recursos disponibles y las necesidades de tu aplicación Spark para garantizar un rendimiento óptimo y la utilización eficiente de recursos.

El parámetro `spark.master` es crucial para configurar adecuadamente la ejecución de Spark y debe ajustarse según el entorno y los requisitos específicos de tu aplicación.

#### **Utilizar `config("spark.master", "local[1]")`**

- **spark = SparkSession.builder.appName("Nombre de mi aplicacion").`config("spark.master", "local[1]")`.enableHiveSupport().getOrCreate()**

El método `config("spark.master", "local[1]")` en la configuración de `SparkSession` indica a Spark cómo debe ejecutarse cuando estás desarrollando y probando tu aplicación localmente. Aquí te explico qué significa esta configuración:

##### `config("spark.master", "local[1]")`

- **Descripción**: Esta configuración especifica el modo de ejecución de Spark. En este caso particular:

  - `spark.master`: Especifica la dirección del "master" (maestro) que Spark debe utilizar para la ejecución. En `local[1]`, `local` indica que Spark se ejecutará en modo local (es decir, sin necesidad de un clúster), y `[1]` indica que se utilizará un solo hilo de procesamiento.

  - `local[1]`: Indica que Spark se ejecutará localmente usando un solo hilo. Esto es útil cuando estás desarrollando y probando tu aplicación Spark en tu propia máquina o en un entorno de desarrollo, donde no necesitas la escalabilidad y la distribución que proporciona un clúster Spark completo.

- **Cuándo utilizarlo**: Utiliza esta configuración cuando estés desarrollando y depurando tu aplicación Spark localmente. Es especialmente útil para pruebas iniciales y desarrollos rápidos, ya que te permite ejecutar tu código de manera rápida sin necesidad de configurar un clúster Spark completo.

##### Ejemplo de Uso:

```python
from pyspark.sql import SparkSession

# Configura y crea la sesión de Spark
spark = SparkSession.builder \
    .appName("Nombre de mi aplicación") \
    .config("spark.master", "local[1]") \
    .enableHiveSupport() \
    .getOrCreate()

# A partir de este punto, puedes usar `spark` para trabajar con DataFrames, ejecutar consultas SQL, etc.
```

En este ejemplo:

- `appName("Nombre de mi aplicación")`: Define el nombre de tu aplicación Spark.
- `config("spark.master", "local[1]")`: Configura Spark para que se ejecute localmente con un solo hilo.
- `enableHiveSupport()`: Opcionalmente, habilita el soporte de Hive si necesitas interactuar con tablas Hive.

##### Consideraciones:

- **Entorno de Desarrollo**: Esta configuración es ideal para entornos de desarrollo donde estás trabajando con cantidades de datos más pequeñas y no necesitas la escalabilidad y la paralelización completa que proporciona un clúster Spark.

- **Producción y Clústeres**: Cuando muevas tu aplicación a un entorno de producción o a un clúster Spark real, deberás ajustar la configuración de `spark.master` para apuntar al gestor de recursos adecuado (como YARN, Mesos, Kubernetes, etc.) y configurar el número adecuado de núcleos y memoria para tus necesidades de procesamiento.

En resumen, `config("spark.master", "local[1]")` es una configuración crucial para ejecutar Spark de manera local y es especialmente útil durante el desarrollo y la depuración de aplicaciones Spark en tu máquina local.